#Tool Calling with below steps
1. Define State
2. Define Tools
3. Bind Tools to LLM
4. Setup Nodes and Graph
5. Implement Conditional edges
6. Rerouting (ReAct pattern)

In [0]:
from dotenv import load_dotenv
import os
import random
from typing import Literal, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from databricks_langchain import ChatDatabricks
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

from langgraph.graph import MessagesState

import mlflow
mlflow.langchain.autolog()

## Step 1: Define State
MessageState is a built in helper to maintain the chat history


In [0]:
class State(MessagesState):
    pass

## Step 2: Define Tools
By giving LLMs access to external APIs or tools, agent can move beyond simple text generation performing actionable tasks

In [0]:
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    """Example:
    >>> multiply(2, 3)
    6
    """
    """Args:
    a: int
    b: int
    """
    """Returns:
    int
    """
    return a * b

## Step 3: Bind tools
Use the bind_tools() method on the LLM, passing an array of functions. Metadata (docstrings) is crucial here, as it tells the agent when and why to use specific tools

In [0]:
llm = ChatDatabricks(endpoint = "databricks-gpt-oss-120b")

llm_with_tools = llm.bind_tools([multiply])

## Step 4: Setup Nodes and Graphs
Create a node for the LLM to call the model and a separate ToolNode to execute the actual function requested by the model

In [0]:
def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [0]:
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode([multiply]))

## Step 5: Implement conditional Edges
Use tools_condition to direct the flow: if the model decides to use a tool, the agent routes to the ToolNode; otherwise, it finishes the task

In [0]:
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)


## Step 6: Rerouting
After the tool runs, the output is passed back to the LLM. If the model still needs more information or further action, it can loop back to the tools (a process known as a ReAct agent) 

In [0]:
builder.add_edge("tools", "tool_calling_llm")

In [0]:
graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage("Hello,what is the weather in Tokyo? and what is 2 * 3?")]})


for i, message in enumerate(result["messages"]):
    print(f"Message {i+1} ({type(message).__name__}): {message.content}")


## Lets add a wikipedia search to get some info


In [0]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

In [0]:
llm = ChatDatabricks(endpoint = "databricks-gpt-oss-120b")

llm_with_tools = llm.bind_tools([wikipedia])

def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode([wikipedia]))


builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage("Tell me about databricks.")]})


for i, message in enumerate(result["messages"]):
    print(f"Message {i+1} ({type(message).__name__}): {message.content}")